In [1]:
import yaml
import logging
import random
import pandas as pd
import numpy as np
import mlflow
import datasets
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from shared import utils

# logging.basicConfig(filename='sentiment_analysis.log', level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# GLOBAL VARIABLES
config_file = "../configs/run1_sentiment.yaml"
with open(config_file, "r") as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)
dataset_config = config.get("dataset")
model_config = config.get("model")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

rng = random.Random(config.get('seed', 0))

logger.info(f"Configs loaded. Device is set to {device}")

In [3]:
experiment_name = config.get("mlflow", dict).get("experiment", None)
if experiment_name is None:
    raise ValueError(
        "Experiment name is not set in the config.\n"
        "Notebook does not run without tracking"
    )
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/13', creation_time=1791120060086, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1791120060086, lifecycle_stage='active', name='llm-bias-variance-2', tags={}, trace_location=None, workspace='default'>

In [4]:
# prep data
dataset=datasets.load_dataset(dataset_config["name"])
split = dataset["train"].train_test_split(train_size=0.7)
train_dataset = split['train']
val_dataset = split['test']
test_dataset = dataset['test']

# Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_config["name"], padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_config["name"]).to(device)
model.eval()

# Generation Config
generation_config = GenerationConfig(**config.get("generation"))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 14247.17it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [5]:
induction_prompt = """You are given a list of inputs and outputs pertaining to the task of classifying the sentiment of a movie review. 
Your task is to create instructions for an LLM to correctly perform zero-shot classification of any movie review based on the patterns in these samples."""
num_train_samples = config.get("train", {}).get("num_samples", [5])
for k in num_train_samples:
    selected_samples = rng.choices(train_dataset, k=k)
    with mlflow.start_run():
        samples_with_layout = "\n\n".join([f"Input: {sample['review']}\nOutput: {sample['label']}" for sample in selected_samples])
        full_prompt = f"## Instruction\n\n{induction_prompt}\n\n## Samples\n\n{samples_with_layout}\n\n### Induction Prompt\n\n" 
        full_prompt_chat_template = [
            {
                "role": "system",
                "content": f"## Instruction\n\n{induction_prompt}\n\nPlace your output under the headline ### Induction Prompt"
            },
            {
                "role": "user",
                "content": f"\n\n## Samples\n\n{samples_with_layout}"
            }
        ]
        
        # tokenize and generate
        if config.get("chat_template", False):
            tokens = tokenizer.apply_chat_template(full_prompt_chat_template, return_tensors="pt") 
        else:
            tokens = tokenizer([full_prompt], return_tensors="pt")
        tokens = tokens.to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(**tokens, generation_config=generation_config)
        output = tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        # Induced Prompt
        generation_chunks = output[0].split("### Induction Prompt")
        if len(generation_chunks) == 1:
            ValueError("No separate induction prompt can be extracted")
        input_prompt = "\n".join(generation_chunks[:-1]) 
        induced_prompt = generation_chunks[-1]
    
        # log params
        mlflow.log_params(dict(num_samples=k, **model_config, **config.get("generation", {})))
    
        # log prompts
        mlflow.log_text(induction_prompt, "induction_prompt.txt")
        mlflow.log_text(input_prompt, "input.txt")
        mlflow.log_text(induced_prompt, "induced_prompt.txt")
        
        # Validation
        val_batch_dataset = val_dataset.batch(
            batch_size=config.get("validation", {}).get("batch_size", 32),
            drop_last_batch=False,
        )
        with torch.no_grad():
            completed_prompt = []
            ground_truth = []
            predictions = []
            for batch in val_batch_dataset:
                # Prepare inputs
                reviews = [
                    [
                        {
                            "role": "system",
                            "content": f"{induced_prompt}\n Place your output under the headline ### Output"
                        },
                        {
                            "role": "user",
                            "content": sample
                        }
                    ]
                    for sample in batch['review']
                ]
                tokens = tokenizer.apply_chat_template(
                    reviews, 
                    return_tensors="pt",
                    padding=True
                ).to(model.device)
        
                # extract outputs
                generated_ids = model.generate(**tokens, generation_config=generation_config)
                full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
                outputs = [
                    utils.coerce_numeric(
                        x.split("### Output")[-1].strip()
                    )
                    for x in full_text
                ]
                ground_truth.extend(batch['label'])
                completed_prompt.extend(full_text)
                predictions.extend(outputs)
        accuracy = (np.array(ground_truth) == np.array(predictions)).mean()
        mlflow.log_metric("accuracy", accuracy)
    
        # Sample traces, stratified on correctness
        results = pd.DataFrame(dict(completed_prompt=completed_prompt, ground_truth=ground_truth, prediction=predictions))
        results["correct"] = results["ground_truth"] == results["prediction"]
        n_traces = config.get("validation", {}).get("num_traces", 5)
        traces = results.groupby("correct", group_keys=False).apply(
            lambda g: g.sample(min(n_traces, len(g)), random_state=config.get("seed", 0))
        )
        mlflow.log_table(traces, "validation_traces.json")

2026/10/08 19:55:32 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/08 19:55:32 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

🏃 View run big-fox-840 at: http://host.docker.internal:5000/#/experiments/13/runs/5b1fb1d55d264eed9d0e7fa23138d8f0
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13
🏃 View run likeable-fox-740 at: http://host.docker.internal:5000/#/experiments/13/runs/e406a785fdca46d9ab32ab6827394bf3
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13


[W1009 01:55:46.283035430 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 859832320 bytes (free: 68222976, total: 25277693952).


🏃 View run chill-shark-200 at: http://host.docker.internal:5000/#/experiments/13/runs/2d3838dff72341479bca42910e6a30cd
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13
🏃 View run likeable-rook-932 at: http://host.docker.internal:5000/#/experiments/13/runs/cc09418d87774acfaf8ae20e8451524e
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13
🏃 View run bouncy-yak-207 at: http://host.docker.internal:5000/#/experiments/13/runs/edad961245c446e1ac41c44e70aea3c5
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13
🏃 View run bittersweet-toad-212 at: http://host.docker.internal:5000/#/experiments/13/runs/f816501c878746e996c8f2a04befd22f
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/13
